In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import sys
import copy
import math
import random
import time
import warnings
import json
import hashlib
import inspect
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, mean_absolute_error, roc_auc_score
from imblearn.over_sampling import RandomOverSampler
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# =====================================================================
# GLOBALS AND HYPERPARAMETERS
# =====================================================================
SEED = 42
DATA_PATH = "/content/drive/MyDrive/RT_IOT2022.csv"
TARGET_COLUMN = "Attack_type"
SELECTED_CLASSES = [
    'MQTT_Publish', 'Thing_Speak', 'Wipro_bulb', 'ARP_poisioning',
    'DDOS_Slowloris', 'DOS_SYN_Hping', 'Metasploit_Brute_Force_SSH',
    'NMAP_FIN_SCAN', 'NMAP_OS_DETECTION', 'NMAP_TCP_scan',
    'NMAP_UDP_SCAN', 'NMAP_XMAS_TREE_SCAN'
]

BATCH_SIZE = 128
TEACHER_EPOCHS = 23
STUDENT_EPOCHS = 30
LEARNING_RATE = 1e-4
PATIENCE = 3

TEACHER_VALID_FRACTION = 0.30
STUDENT_VALID_FRACTION = 0.20
TEACHER_POOL_FRACTION = 0.80
STUDENT_PRIVATE_FRACTION = 0.00
PUBLIC_FRACTION = 0.05
FINAL_TEST_FRACTION = 0.15

LAPLACE_INVERSE_SCALE = 1.3
DELTA = 1e-5
MAX_MOMENT = 32
USE_OVERSAMPLING = True
OUTPUT_DIR = "./pate_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =====================================================================
# DATASET LOADING AND PIPELINE SPLITTING
# =====================================================================
def load_dataset(path, target_column, selected_classes):
    data = pd.read_csv(path, index_col=0)
    if target_column not in data.columns:
        raise ValueError(f"Target column '{target_column}' was not found.")
    data = data[data[target_column].isin(selected_classes)].copy()
    if len(data) == 0:
        raise ValueError("No observations remain after class filtering.")
    return data

def preprocess_and_split(data, target_column, seed=42):
    data = data.copy()
    total_fraction = TEACHER_POOL_FRACTION + STUDENT_PRIVATE_FRACTION + PUBLIC_FRACTION + FINAL_TEST_FRACTION
    if not np.isclose(total_fraction, 1.0):
        raise ValueError("Dataset fractions must sum to 1.0.")

    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(data[target_column])
    X = data.drop(columns=[target_column])

    non_numeric = X.select_dtypes(include=["object", "category"]).columns.tolist()
    if len(non_numeric) > 0:
        X = X.drop(columns=non_numeric)

    X = X.apply(pd.to_numeric, errors="coerce")
    valid_rows = ~X.isna().any(axis=1)
    X = X.loc[valid_rows].reset_index(drop=True)
    y = y[valid_rows.to_numpy()]

    X = X.to_numpy(dtype=np.float32)
    y = np.asarray(y, dtype=np.int64)

    X_teacher, X_heldout, y_teacher, y_heldout = train_test_split(
        X, y, test_size=(1.0 - TEACHER_POOL_FRACTION), random_state=seed, stratify=y
    )

    heldout_total_sub_fraction = STUDENT_PRIVATE_FRACTION + PUBLIC_FRACTION + FINAL_TEST_FRACTION
    test_size_final_test = FINAL_TEST_FRACTION / heldout_total_sub_fraction

    X_public, X_final_test, y_public, y_final_test = train_test_split(
        X_heldout, y_heldout, test_size=test_size_final_test, random_state=seed, stratify=y_heldout
    )

    X_private = np.zeros((0, X_teacher.shape[1]), dtype=np.float32)
    y_private = np.zeros((0,), dtype=np.int64)

    return X_teacher, y_teacher, X_private, y_private, X_public, y_public, X_final_test, y_final_test, label_encoder

class TabularDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        return len(self.features)
    def __getitem__(self, index):
        return self.features[index], self.labels[index]

# =====================================================================
# MODEL ARCHITECTURES (TEACHERS AND STUDENTS)
# =====================================================================
class TeacherLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1, num_classes=3):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)
    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        _, (hidden, _) = self.lstm(x)
        return F.log_softmax(self.fc(hidden[-1]), dim=1)

class TeacherGRU(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1, num_classes=3):
        super().__init__()
        self.gru = nn.GRU(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)
    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        _, hidden = self.gru(x)
        return F.log_softmax(self.fc(hidden[-1]), dim=1)

class TeacherCNN(nn.Module):
    def __init__(self, input_size, num_classes=3):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(64 * input_size, 128)
        self.fc2 = nn.Linear(128, num_classes)
    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = torch.flatten(x, start_dim=1)
        x = F.relu(self.fc1(x))
        return F.log_softmax(self.fc2(x), dim=1)

class TeacherMLP(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2, num_classes=3):
        super().__init__()
        layers = []
        current_size = input_size
        for _ in range(num_layers):
            layers.append(nn.Linear(current_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.20))
            current_size = hidden_size
        self.network = nn.Sequential(*layers)
        self.output_layer = nn.Linear(hidden_size, num_classes)
    def forward(self, x):
        if x.dim() == 3:
            x = x.mean(dim=1)
        return F.log_softmax(self.output_layer(self.network(x)), dim=1)

class StudentMLP(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2, num_classes=3):
        super().__init__()
        layers = []
        current_size = input_size
        for _ in range(num_layers):
            layers.append(nn.Linear(current_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.20))
            current_size = hidden_size
        self.network = nn.Sequential(*layers)
        self.output_layer = nn.Linear(hidden_size, num_classes)
    def forward(self, x):
        if x.dim() == 3:
            x = x.mean(dim=1)
        return F.log_softmax(self.output_layer(self.network(x)), dim=1)

class StudentLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1, num_classes=3):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)
    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        _, (hidden, _) = self.lstm(x)
        return F.log_softmax(self.fc(hidden[-1]), dim=1)

class StudentGRU(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1, num_classes=3):
        super().__init__()
        self.gru = nn.GRU(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)
    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        _, hidden = self.gru(x)
        return F.log_softmax(self.fc(hidden[-1]), dim=1)

class StudentCNN(nn.Module):
    def __init__(self, input_size, num_classes=3):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(64 * input_size, 128)
        self.fc2 = nn.Linear(128, num_classes)
    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = torch.flatten(x, start_dim=1)
        x = F.relu(self.fc1(x))
        return F.log_softmax(self.fc2(x), dim=1)

# =====================================================================
# MODEL BUILD FACTORIES
# =====================================================================
def build_teacher(architecture, input_size, num_classes):
    if architecture == "LSTM": return TeacherLSTM(input_size=input_size, num_classes=num_classes)
    if architecture == "GRU": return TeacherGRU(input_size=input_size, num_classes=num_classes)
    if architecture == "CNN": return TeacherCNN(input_size=input_size, num_classes=num_classes)
    if architecture == "MLP": return TeacherMLP(input_size=input_size, num_classes=num_classes)
    raise ValueError(f"Unknown teacher architecture: {architecture}")

def build_student(architecture, input_size, num_classes):
    if architecture == "MLP": return StudentMLP(input_size=input_size, num_classes=num_classes)
    if architecture == "LSTM": return StudentLSTM(input_size=input_size, num_classes=num_classes)
    if architecture == "GRU": return StudentGRU(input_size=input_size, num_classes=num_classes)
    if architecture == "CNN": return StudentCNN(input_size=input_size, num_classes=num_classes)
    raise ValueError(f"Unknown student architecture: {architecture}")

# =====================================================================
# UTILITY TRAINING AND RIGOROUS EVALUATION FUNCTIONS
# =====================================================================
def train_model(model, train_loader, valid_loader, epochs, learning_rate, patience, device):
    model = model.to(device)
    criterion = nn.NLLLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    best_validation_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())
    epochs_without_improvement = 0
    history = []

    for epoch in range(epochs):
        model.train()
        total_train_loss, total_train, correct_train = 0.0, 0, 0
        for is_input, labels in train_loader:
            is_input, labels = is_input.to(device), labels.to(device)
            optimizer.zero_grad()
            log_probabilities = model(is_input)
            loss = criterion(log_probabilities, labels)
            loss.backward()
            optimizer.step()
            b_size = labels.size(0)
            total_train_loss += loss.item() * b_size
            predictions = torch.argmax(log_probabilities, dim=1)
            correct_train += (predictions == labels).sum().item()
            total_train += b_size

        model.eval()
        total_valid_loss, total_valid, correct_valid = 0.0, 0, 0
        with torch.no_grad():
            for is_input, labels in valid_loader:
                is_input, labels = is_input.to(device), labels.to(device)
                log_probabilities = model(is_input)
                loss = criterion(log_probabilities, labels)
                b_size = labels.size(0)
                total_valid_loss += loss.item() * b_size
                predictions = torch.argmax(log_probabilities, dim=1)
                correct_valid += (predictions == labels).sum().item()
                total_valid += b_size

        valid_loss = total_valid_loss / total_valid
        history.append({
            "epoch": epoch + 1,
            "train_loss": total_train_loss / total_train,
            "valid_loss": valid_loss,
            "train_accuracy": correct_train / total_train,
            "valid_accuracy": correct_valid / total_valid
        })

        if valid_loss < best_validation_loss:
            best_validation_loss = valid_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
        if epochs_without_improvement >= patience:
            break

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)

def predict_labels(model, loader, device):
    model.eval()
    predictions = []
    with torch.no_grad():
        for inputs, _ in loader:
            inputs = inputs.to(device)
            log_probabilities = model(inputs)
            batch_predictions = torch.argmax(log_probabilities, dim=1)
            predictions.extend(batch_predictions.cpu().numpy())
    return np.asarray(predictions, dtype=np.int64)

def evaluate_student_rigorous(student_model, X_test, y_test, num_classes, batch_size, device):
    student_model.eval()
    loader = DataLoader(TabularDataset(X_test, y_test), batch_size=batch_size, shuffle=False)
    all_predictions = []
    all_probs = []

    with torch.no_grad():
        for inputs, _ in loader:
            inputs = inputs.to(device)
            log_probabilities = student_model(inputs)
            probs = torch.exp(log_probabilities)
            batch_predictions = torch.argmax(log_probabilities, dim=1)
            all_predictions.extend(batch_predictions.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    all_predictions = np.array(all_predictions)
    all_probs = np.array(all_probs)

    accuracy = accuracy_score(y_test, all_predictions)
    precision = precision_score(y_test, all_predictions, average='macro', zero_division=0)
    recall = recall_score(y_test, all_predictions, average='macro', zero_division=0)
    f1_macro = f1_score(y_test, all_predictions, average='macro', zero_division=0)

    y_test_bin = label_binarize(y_test, classes=np.arange(num_classes))
    try:
        if num_classes == 2:
            auc = roc_auc_score(y_test, all_probs[:, 1])
        else:
            auc = roc_auc_score(y_test_bin, all_probs, average='macro', multi_class='ovr')
    except Exception:
        auc = np.nan

    try:
        mae = mean_absolute_error(y_test_bin, all_probs)
    except Exception:
        mae = np.nan

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1_macro,
        "auc": auc,
        "mae": mae
    }

# =====================================================================
# PATE PRIVATE ACCOUNTING AND VOTING MECHANISMS
# =====================================================================
def teacher_predictions_to_counts(teacher_preds, num_classes):
    teacher_preds = np.asarray(teacher_preds, dtype=np.int64)
    num_teachers, num_queries = teacher_preds.shape
    counts = np.zeros((num_queries, num_classes), dtype=np.int64)
    for query_index in range(num_queries):
        counts[query_index] = np.bincount(teacher_preds[:, query_index], minlength=num_classes)
    return counts

def aggregate_teacher_votes_laplace(teacher_preds, num_classes, laplace_inverse_scale, random_seed=None):
    counts = teacher_predictions_to_counts(teacher_preds, num_classes)
    rng = np.random.default_rng(random_seed)
    laplace_scale = 1.0 / laplace_inverse_scale
    noise = rng.laplace(loc=0.0, scale=laplace_scale, size=counts.shape)
    noisy_counts = counts.astype(np.float64) + noise
    released_labels = np.argmax(noisy_counts, axis=1)
    return released_labels, counts, noisy_counts

def logaddexp_scalar(a, b):
    if a == -math.inf: return b
    if b == -math.inf: return a
    maximum = max(a, b)
    return maximum + math.log(math.exp(a - maximum) + math.exp(b - maximum))

def logmgf_exact(q, mechanism_eps, moment_order):
    lam = float(moment_order)
    eps = float(mechanism_eps)
    bound_1 = 0.5 * (eps ** 2) * lam * (lam + 1.0)
    bound_3 = eps * lam
    bound_2 = math.inf
    denominator = 1.0 - math.exp(eps) * q
    if q < 0.5 and denominator > 0.0:
        log_one_minus_q = math.log1p(-q)
        log_term_1 = log_one_minus_q + lam * (log_one_minus_q - math.log(denominator))
        log_term_2 = math.log(q) + eps * lam if q > 0.0 else -math.inf
        bound_2 = logaddexp_scalar(log_term_1, log_term_2)
    return min(bound_1, bound_2, bound_3)

def compute_q_noisy_max(counts, laplace_inverse_scale):
    counts = np.asarray(counts, dtype=np.float64)
    num_classes = len(counts)
    if num_classes < 2:
        return 0.0
    winner = int(np.argmax(counts))
    winning_votes = counts[winner]
    competitors = np.delete(counts, winner)
    gaps = laplace_inverse_scale * (winning_votes - competitors)
    q = float(np.sum((gaps + 2.0) * np.exp(-gaps) / 4.0))
    return min(max(q, 0.0), 1.0 - 1.0 / num_classes)

def data_dependent_log_moment(counts, laplace_inverse_scale, moment_order):
    q = compute_q_noisy_max(counts, laplace_inverse_scale)
    mechanism_eps = 2.0 * laplace_inverse_scale
    return logmgf_exact(q, mechanism_eps, moment_order)

def data_independent_log_moment(laplace_inverse_scale, moment_order):
    mechanism_eps = 2.0 * laplace_inverse_scale
    lam = float(moment_order)
    concentration_bound = 0.5 * (mechanism_eps ** 2) * lam * (lam + 1.0)
    direct_dp_bound = mechanism_eps * lam
    return min(concentration_bound, direct_dp_bound)

def calculate_pate_privacy(vote_counts, laplace_inverse_scale=1.3, delta=1e-5, moments=32):
    vote_counts = np.asarray(vote_counts, dtype=np.int64)
    num_queries, classes_dim = vote_counts.shape
    orders = np.arange(1, moments + 1, dtype=np.float64)
    dependent_total = np.zeros(moments, dtype=np.float64)
    independent_total = np.zeros(moments, dtype=np.float64)
    q_values, vote_margins = [], []

    for counts in vote_counts:
        q = compute_q_noisy_max(counts, laplace_inverse_scale)
        q_values.append(q)
        sorted_counts = np.sort(counts)
        vote_margins.append(sorted_counts[-1] - sorted_counts[-2])
        for pos, order in enumerate(orders):
            dependent_total[pos] += data_dependent_log_moment(counts, laplace_inverse_scale, order)
            independent_total[pos] += data_independent_log_moment(laplace_inverse_scale, order)

    dep_candidates = (dependent_total - math.log(delta)) / orders
    ind_candidates = (independent_total - math.log(delta)) / orders
    dep_idx = np.argmin(dep_candidates)
    ind_idx = np.argmin(ind_candidates)

    dep_epsilon = float(dep_candidates[dep_idx])
    ind_epsilon = float(ind_candidates[ind_idx])

    if dep_epsilon > ind_epsilon + 1e-10:
        raise AssertionError(
            f"Invalid privacy result: dependent epsilon "
            f"{dep_epsilon} exceeds independent epsilon {ind_epsilon}"
        )

    return {
        "data_dependent_epsilon": dep_epsilon,
        "data_independent_epsilon": ind_epsilon,
        "q_values": np.asarray(q_values),
        "vote_margins": np.asarray(vote_margins)
    }

# =====================================================================
# COMPLIANCE UNIT TESTS
# =====================================================================
def test_dd_never_exceeds_di():
    test_votes = np.array([
        [9, 0, 0],
        [6, 2, 1],
        [4, 3, 2],
        [3, 3, 3]
    ])
    result = calculate_pate_privacy(
        test_votes,
        laplace_inverse_scale=1.3,
        delta=1e-5,
        moments=32
    )
    assert (
        result["data_dependent_epsilon"]
        <= result["data_independent_epsilon"] + 1e-10
    ), result

def run_accounting_tests():
    print("\n" + "=" * 70)
    print("RUNNING PATE ACCOUNTING COMPLIANCE TESTS")
    print("=" * 70)
    counts = np.array([10, 6, 2], dtype=float)
    eta = 0.1
    gaps = eta * np.array([4.0, 8.0])
    expected_q = min(np.sum((gaps + 2.0) * np.exp(-gaps) / 4.0), 2.0 / 3.0)
    actual_q = compute_q_noisy_max(counts, eta)
    assert np.isclose(actual_q, expected_q), f"q discrepancy: {actual_q} vs {expected_q}"
    print("PASS: Direct multiclass q verification")

    di_test_val = data_independent_log_moment(laplace_inverse_scale=1.0, moment_order=2)
    assert np.isclose(di_test_val, 4.0), f"DI log moment discrepancy: {di_test_val} vs 4.0"
    print("PASS: Independent hand-calculated DI regression test")

    # Run additional target verification test
    test_dd_never_exceeds_di()
    print("PASS: Data-Dependent <= Data-Independent invariant test (test_dd_never_exceeds_di)")
    print("All core compliance unit tests passed successfully!")

# =====================================================================
# MASTER LIFECYCLE PIPELINE EXPERIMENT RUNNER
# =====================================================================
def run_custom_pate_experiment(config):
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    data = load_dataset(DATA_PATH, TARGET_COLUMN, SELECTED_CLASSES)
    X_teacher, y_teacher, X_private, y_private, X_public, y_public, X_final_test, y_final_test, label_encoder = preprocess_and_split(data, TARGET_COLUMN, seed=SEED)

    input_size = X_teacher.shape[1]
    num_classes = len(np.unique(y_teacher))

    scaler = StandardScaler()
    X_teacher_scaled = scaler.fit_transform(X_teacher)
    X_public_scaled = scaler.transform(X_public)
    X_final_test_scaled = scaler.transform(X_final_test)

    n = len(X_teacher_scaled)
    indices = np.arange(n)
    rng = np.random.default_rng(SEED)
    rng.shuffle(indices)
    partitions = np.array_split(indices, config["num_teachers"])

    train_loaders, valid_loaders = [], []
    for teacher_id, teacher_indices in enumerate(partitions):
        X_part = X_teacher_scaled[teacher_indices]
        y_part = y_teacher[teacher_indices]
        try:
            X_train, X_valid, y_train, y_valid = train_test_split(X_part, y_part, test_size=TEACHER_VALID_FRACTION, random_state=SEED + teacher_id, stratify=y_part)
        except ValueError:
            X_train, X_valid, y_train, y_valid = train_test_split(X_part, y_part, test_size=TEACHER_VALID_FRACTION, random_state=SEED + teacher_id, stratify=None)
        if USE_OVERSAMPLING:
            sampler = RandomOverSampler(random_state=SEED + teacher_id)
            X_train, y_train = sampler.fit_resample(X_train, y_train)
        train_loaders.append(DataLoader(TabularDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True))
        valid_loaders.append(DataLoader(TabularDataset(X_valid, y_valid), batch_size=BATCH_SIZE, shuffle=False))

    teachers = []
    for idx in range(config["num_teachers"]):
        arch = config["teacher_archs"][idx % len(config["teacher_archs"])]
        model = build_teacher(arch, input_size, num_classes)
        trained_model, _ = train_model(model, train_loaders[idx], valid_loaders[idx], TEACHER_EPOCHS, LEARNING_RATE, PATIENCE, DEVICE)
        teachers.append(trained_model)

    public_loader = DataLoader(TabularDataset(X_public_scaled, y_public), batch_size=BATCH_SIZE, shuffle=False)
    all_teacher_preds = []
    for idx, t_model in enumerate(teachers):
        preds = predict_labels(t_model, public_loader, DEVICE)
        all_teacher_preds.append(preds)
    all_teacher_preds = np.stack(all_teacher_preds)

    released_labels, vote_counts, _ = aggregate_teacher_votes_laplace(all_teacher_preds, num_classes, LAPLACE_INVERSE_SCALE, random_seed=SEED)
    privacy_results = calculate_pate_privacy(vote_counts, LAPLACE_INVERSE_SCALE, DELTA, MAX_MOMENT)

    mean_vote_margin = float(np.mean(privacy_results["vote_margins"]))

    X_student_train, X_student_valid, y_student_train, y_student_valid = train_test_split(X_public_scaled, released_labels, test_size=STUDENT_VALID_FRACTION, random_state=SEED, stratify=released_labels)
    student_train_loader = DataLoader(TabularDataset(X_student_train, y_student_train), batch_size=BATCH_SIZE, shuffle=True)
    student_valid_loader = DataLoader(TabularDataset(X_student_valid, y_student_valid), batch_size=BATCH_SIZE, shuffle=False)

    student_model = build_student(config["student_arch"], input_size, num_classes)
    trained_student, _ = train_model(student_model, student_train_loader, student_valid_loader, STUDENT_EPOCHS, LEARNING_RATE, PATIENCE, DEVICE)

    student_metrics = evaluate_student_rigorous(trained_student, X_final_test_scaled, y_final_test, num_classes, BATCH_SIZE, DEVICE)

    return {
        "dep_eps": privacy_results["data_dependent_epsilon"],
        "ind_eps": privacy_results["data_independent_epsilon"],
        "mean_vote_margin": mean_vote_margin,
        **student_metrics
    }

# =====================================================================
# ALL 15 SPECIFIC ARCHITECTURAL CONFIGURATIONS
# =====================================================================
all_15_configurations = [
    {"name": "Config 1 (1 LSTM Teacher -> 1 GRU Student)", "num_teachers": 1, "teacher_archs": ["LSTM"], "student_arch": "GRU"},
    {"name": "Config 2 (3 LSTM, 3 CNN, 3 MLP Teachers -> 1 LSTM Student)", "num_teachers": 9, "teacher_archs": ["LSTM"]*3 + ["CNN"]*3 + ["MLP"]*3, "student_arch": "LSTM"},
    {"name": "Config 3 (3 LSTM, 3 MLP, 3 CNN Teachers -> 1 GRU Student)", "num_teachers": 9, "teacher_archs": ["LSTM"]*3 + ["MLP"]*3 + ["CNN"]*3, "student_arch": "GRU"},
    {"name": "Config 4 (3 LSTM Teachers -> 1 LSTM Student)", "num_teachers": 3, "teacher_archs": ["LSTM"]*3, "student_arch": "LSTM"},
    {"name": "Config 5 (1 LSTM, 1 GRU, 1 CNN Teacher -> 1 MLP Student)", "num_teachers": 3, "teacher_archs": ["LSTM", "GRU", "CNN"], "student_arch": "MLP"},
    {"name": "Config 6 (3 MLP Teachers -> 1 MLP Student)", "num_teachers": 3, "teacher_archs": ["MLP"]*3, "student_arch": "MLP"},
    {"name": "Config 7 (3 CNN Teachers -> 1 CNN Student)", "num_teachers": 3, "teacher_archs": ["CNN"]*3, "student_arch": "CNN"},
    {"name": "Config 8 (1 MLP, 1 CNN, 1 GRU, 1 LSTM Teacher -> 1 CNN Student)", "num_teachers": 4, "teacher_archs": ["MLP", "CNN", "GRU", "LSTM"], "student_arch": "CNN"},
    {"name": "Config 9 (1 LSTM Teacher -> 1 LSTM Student)", "num_teachers": 1, "teacher_archs": ["LSTM"], "student_arch": "LSTM"},
    {"name": "Config 10 (6 LSTM Teachers -> 1 LSTM Student)", "num_teachers": 6, "teacher_archs": ["LSTM"]*6, "student_arch": "LSTM"},
    {"name": "Config 11 (6 CNN, 6 GRU, 6 LSTM Teachers -> 1 CNN Student)", "num_teachers": 18, "teacher_archs": ["CNN"]*6 + ["GRU"]*6 + ["LSTM"]*6, "student_arch": "CNN"},
    {"name": "Config 12 (3 LSTM, 3 GRU, 3 CNN Teachers -> 1 MLP Student)", "num_teachers": 9, "teacher_archs": ["LSTM"]*3 + ["GRU"]*3 + ["CNN"]*3, "student_arch": "MLP"},
    {"name": "Config 13 (3 LSTM, 4 MLP, 3 CNN Teachers -> 1 GRU Student)", "num_teachers": 10, "teacher_archs": ["LSTM"]*3 + ["MLP"]*4 + ["CNN"]*3, "student_arch": "GRU"},
    {"name": "Config 14 (10 CNN Teachers -> 1 CNN Student)", "num_teachers": 10, "teacher_archs": ["CNN"]*10, "student_arch": "CNN"},
    {"name": "Config 15 (1 LSTM, 1 MLP, 1 CNN Teachers -> 1 GRU Student)", "num_teachers": 3, "teacher_archs": ["LSTM", "MLP", "CNN"], "student_arch": "GRU"}
]

# =====================================================================
# RUN SYSTEMATIC SIMULATION PIPELINE
# =====================================================================
if not os.path.exists(DATA_PATH):
    print(f"[ERROR] File '{DATA_PATH}' is missing. Please mount Drive or upload the dataset.")
else:
    print("Path validated. Beginning unified systematic execution of all 15 scenarios...")

    # Run direct multiclass accounting tests including the new boundary inequality check
    run_accounting_tests()

    final_unified_table = []
    for idx, config in enumerate(all_15_configurations, 1):
        print(f"\n[{idx}/15] Active Execution: {config['name']}...")
        try:
            exp_results = run_custom_pate_experiment(config)
            exp_results["Configuration"] = config["name"]
            final_unified_table.append(exp_results)
            print(f"SUCCESS: Completed config {idx} successfully.")
        except Exception as e:
            print(f"FAILURE: Config {config['name']} halted. Error: {str(e)}")

    if len(final_unified_table) > 0:
        df_all_results = pd.DataFrame(final_unified_table)
        cols_order = ["Configuration", "dep_eps", "ind_eps", "accuracy", "precision", "recall", "f1_score", "auc", "mae", "mean_vote_margin"]
        df_all_results = df_all_results[[c for c in cols_order if c in df_all_results.columns]]

        # Export results directly to CSV instead of manual copying
        csv_output_path = os.path.join(OUTPUT_DIR, "all_configurations_summary.csv")
        df_all_results.to_csv(csv_output_path, index=False)
        print(f"\n[SYSTEM] Results successfully exported to CSV at: {csv_output_path}")

        display(df_all_results)
    else:
        print("No experiments completed.")

Path validated. Beginning unified systematic execution of all 15 scenarios...

RUNNING PATE ACCOUNTING COMPLIANCE TESTS
PASS: Direct multiclass q verification
PASS: Independent hand-calculated DI regression test
PASS: Data-Dependent <= Data-Independent invariant test (test_dd_never_exceeds_di)
All core compliance unit tests passed successfully!

[1/15] Active Execution: Config 1 (1 LSTM Teacher -> 1 GRU Student)...
SUCCESS: Completed config 1 successfully.

[2/15] Active Execution: Config 2 (3 LSTM, 3 CNN, 3 MLP Teachers -> 1 LSTM Student)...
SUCCESS: Completed config 2 successfully.

[3/15] Active Execution: Config 3 (3 LSTM, 3 MLP, 3 CNN Teachers -> 1 GRU Student)...
SUCCESS: Completed config 3 successfully.

[4/15] Active Execution: Config 4 (3 LSTM Teachers -> 1 LSTM Student)...
SUCCESS: Completed config 4 successfully.

[5/15] Active Execution: Config 5 (1 LSTM, 1 GRU, 1 CNN Teacher -> 1 MLP Student)...
SUCCESS: Completed config 5 successfully.

[6/15] Active Execution: Config 6 (

,Configuration,dep_eps,ind_eps,accuracy,precision,recall,f1_score,auc,mae,mean_vote_margin
0,Config 1 (1 LSTM Teacher -> 1 GRU Student),16005.959779,16005.959779,0.910819,0.565564,0.461523,0.462013,0.891426,0.122059,1.000000
1,"Config 2 (3 LSTM, 3 CNN, 3 MLP Teachers -> 1 L...",189.483229,16005.959779,0.940817,0.619957,0.577664,0.577917,0.976492,0.023047,8.872807
2,"Config 3 (3 LSTM, 3 MLP, 3 CNN Teachers -> 1 G...",161.091888,16005.959779,0.970110,0.671370,0.623723,0.625540,0.980138,0.018304,8.902372
3,Config 4 (3 LSTM Teachers -> 1 LSTM Student),16005.959779,16005.959779,0.939030,0.600774,0.569214,0.569186,0.967311,0.044530,2.985705
4,"Config 5 (1 LSTM, 1 GRU, 1 CNN Teacher -> 1 ML...",16005.959779,16005.959779,0.962638,0.606200,0.603759,0.600783,0.955713,0.039509,2.982943
5,Config 6 (3 MLP Teachers -> 1 MLP Student),16005.959779,16005.959779,0.962476,0.772586,0.640780,0.659275,0.938895,0.039914,2.972547
6,Config 7 (3 CNN Teachers -> 1 CNN Student),16005.959779,16005.959779,0.972493,0.699470,0.715201,0.697446,0.970265,0.034711,2.959227
7,"Config 8 (1 MLP, 1 CNN, 1 GRU, 1 LSTM Teacher ...",16005.959779,16005.959779,0.975363,0.711110,0.723144,0.708372,0.972364,0.018805,3.977583
8,Config 9 (1 LSTM Teacher -> 1 LSTM Student),16005.959779,16005.959779,0.906541,0.408587,0.437494,0.416091,0.924594,0.123757,1.000000
9,Config 10 (6 LSTM Teachers -> 1 LSTM Student),1715.991543,16005.959779,0.940546,0.585570,0.573061,0.571269,0.976219,0.023626,5.965075
